# StrataRL: Kaggle Execution Pipeline

This notebook sets up the environment and executes the StrataRL training loop on Kaggle (P100 / T4). It addresses dependency conflicts natively found in the Kaggle environment.

## Step 1: Copy Source Code
Kaggle Dataset inputs are read-only. We copy the repository to `/kaggle/working` to allow script execution, model saving, and logging.

In [ ]:
from google.colab import drive
import os, shutil
drive.mount('/content/drive')

# Assuming you uploaded the code as StrataRL-main.zip to your Colab root or Drive
if not os.path.exists('/content/StrataRL-main'):
    !unzip -q /content/drive/MyDrive/StrataRL-main.zip -d /content/
print("✓ Repository ready")


## Step 2: Environment Setup
Configure PYTHONPATH and fix directory structure (e.g. missing `__init__.py` files).

In [ ]:
import sys
PROJECT_ROOT = "/content/StrataRL-main"
os.environ["PYTHONPATH"] = PROJECT_ROOT
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["DISABLE_TRANSFORMERS_AV"] = "1"
os.environ["BNB_CUDA_VERSION"] = "121"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("✓ Working directory:", os.getcwd())

folders = ["training", "rewards", "monitoring", "curriculum", "eval", "data", "m4", "engines"]
for folder in folders:
    path = os.path.join(PROJECT_ROOT, folder, "__init__.py")
    if not os.path.exists(path):
        open(path, "w").close()
print("✓ __init__.py files ensured")

## Step 3: Clean and Install Dependencies
Kaggle environments come with many conflicting libraries pre-installed. We purge the conflicting ones and install the strict versions required for Qwen 2.5 and vLLM-style generation.

In [ ]:
%%bash
pip uninstall -y torchvision bitsandbytes transformers accelerate peft trl torchao

pip install -q \
torch==2.4.1+cu121 \
torchaudio==2.4.1+cu121 \
--index-url https://download.pytorch.org/whl/cu121

pip install -q \
transformers==4.46.3 \
datasets \
accelerate \
peft \
trl \
sympy \
wandb \
sentencepiece \
protobuf \
scipy \
scikit-learn \
pytest

## Step 4: Fix BitsAndBytes
The bitsandbytes version can be notoriously finicky in Kaggle. We force install 0.44.0 which has the required pre-compiled CUDA kernels.

In [ ]:
import subprocess
subprocess.run(["pip", "uninstall", "-y", "bitsandbytes"], check=False)
subprocess.run(["pip", "install", "bitsandbytes==0.44.0", "-q"], check=True)
print("✓ bitsandbytes 0.44.0 installed")

## Step 5: GPU & Imports Check
Verify CUDA is accessible and our core custom modules can be imported.

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

cmd = """
from rewards.reward_engine import *
from rewards.structural_reward import *
from training.advantage import *
from training.policy_update import *
from monitoring.monitor import *
print("✓ Core modules imported successfully")
"""
r = subprocess.run(["python", "-c", cmd], capture_output=True, text=True, env={**os.environ, "PYTHONPATH": PROJECT_ROOT})
print(r.stdout or r.stderr)

## Step 6: Weights & Biases Authentication
Retrieves the `WANDB_API_KEY` from your Kaggle Secrets.

In [ ]:
import os
import wandb
from google.colab import userdata

try:
    wandb_api_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_api_key)
    print("✓ wandb logged in")
except Exception as e:
    print("⚠️ Could not authenticate with Weights & Biases.")
    os.environ["WANDB_DISABLED"] = "true"


## Step 7 (Optional): Run Test Suite
Runs our unit tests to ensure nothing broke during environment setup.

In [ ]:
%%bash
cd /content/StrataRL-main
export PYTHONPATH=.
pytest tests/ -v --tb=short

## Step 8: Execution Configuration & Launch
Spawns the training script as a subprocess. We use a subprocess so that stdout (training logs) streams cleanly to the notebook output in real-time.

In [ ]:
import subprocess
import os
import sys
import time

# Record training start time for time-budget calculations
NOTEBOOK_START_TIME = time.time()

env = {
    **os.environ,
    "PYTHONPATH": PROJECT_ROOT,
    "PYTHONUNBUFFERED": "1",
    "BNB_CUDA_VERSION": "121",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    # Uncomment below to disable W&B logging for testing
    # "WANDB_DISABLED": "true",
    # "WANDB_MODE": "disabled",
}

print("Starting Kaggle StrataRL run...\n")

process = subprocess.Popen(
    [
        "python",
        "-u",
        "training/train.py",
        "--config",
        "configs/exp_02_condition_b.yaml"
    ],
    cwd=PROJECT_ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

try:
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()
except KeyboardInterrupt:
    print("\nStopping training gracefully...")
    process.terminate()

process.wait()
print(f"\nTraining finished with exit code: {process.returncode}")

## Step 9: Final Evaluation

Runs evaluation against the trained adapter. Dynamically adjusts N based on remaining time budget to avoid Kaggle timeout.

In [ ]:
import subprocess, os, sys, json, time
from pathlib import Path

PROJECT_ROOT = "/content/StrataRL-main"
env = {**os.environ, "PYTHONPATH": PROJECT_ROOT, "PYTHONUNBUFFERED": "1"}

# Time-budget awareness: Kaggle kills at 43200s (12h)
KAGGLE_TIMEOUT = 43200
EVAL_SAFETY_MARGIN = 600  # 10 min safety buffer
try:
    elapsed = time.time() - NOTEBOOK_START_TIME
except NameError:
    # Fallback if NOTEBOOK_START_TIME not defined (e.g. running cell independently)
    elapsed = 8 * 3600
remaining = KAGGLE_TIMEOUT - elapsed - EVAL_SAFETY_MARGIN
# With batched eval: ~8s per sample per benchmark (3 benchmarks)
max_samples = max(10, int(remaining / (8 * 3)))
n_eval = min(max_samples, 50)
print(f"Elapsed: {elapsed/3600:.1f}h | Remaining: {remaining/60:.0f}min | Using N={n_eval} samples per benchmark\n")

adapter_path = Path(PROJECT_ROOT) / "outputs" / "final"
if not adapter_path.exists():
    ckpts = sorted((Path(PROJECT_ROOT) / "outputs").glob("step_*"), key=lambda p: int(p.name.split("_")[-1]))
    if ckpts:
        adapter_path = ckpts[-1]
        print(f"outputs/final not found — using latest checkpoint: {adapter_path}")
    else:
        raise RuntimeError("No trained adapter found under outputs/")

eval_script = Path(PROJECT_ROOT) / "scripts" / "evaluate_adapter.py"
if eval_script.exists():
    cmd = [
        "python", "-u", "scripts/evaluate_adapter.py",
        "--adapter_path", str(adapter_path),
        "--n_samples", str(n_eval),
        "--output", "reports/final_eval.json",
    ]
else:
    print("scripts/evaluate_adapter.py not found — falling back to measure_baseline.py")
    cmd = [
        "python", "-u", "scripts/measure_baseline.py",
        "--model", str(adapter_path),
        "--n_samples", str(n_eval),
        "--benchmarks", "gsm8k", "mmlu", "strategyqa",
        "--baseline_path", "reports/actual_baselines.json",
        "--output", "reports/final_eval.json",
    ]

# Use Popen with streaming output to avoid timeout
process = subprocess.Popen(
    cmd, cwd=PROJECT_ROOT, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

process.wait()
print(f"\nEvaluation finished with exit code: {process.returncode}")

final_path = Path(PROJECT_ROOT) / "reports" / "final_eval.json"
if final_path.exists():
    print("\n✓ Final eval written to", final_path)
    data = json.loads(final_path.read_text())
    print(json.dumps(data, indent=2))

    baseline_path = Path(PROJECT_ROOT) / "reports" / "actual_baselines.json"
    if baseline_path.exists():
        baselines = json.loads(baseline_path.read_text())
        print("\n── Delta vs measured baseline ──────────────────────────────────")
        for bm in ["gsm8k", "mmlu", "strategyqa"]:
            b = baselines.get(bm)
            f = data.get(bm)
            if b is not None and f is not None:
                print(f"  {bm:<12} baseline={b:.3f}  final={f:.3f}  delta={f-b:+.3f}")
else:
    print("✗ final_eval.json not produced — check output above")